# Step 2 — Fine-Tuning with LoRA & MoRA

**Model**: `google/roberta2roberta_L-24_cnn_daily_mail`  
**Task**: Abstractive Summarisation (CNN/DailyMail)  

## What we do in this notebook
1. Load the CSV chunk downloaded in Step 1  
2. Build a `torch.utils.data.Dataset` and `DataLoader`  
3. Wrap the HuggingFace model in `nn.Module`  
4. Fine-tune with **LoRA** (via `peft`) — adapts rank-decomposed matrices  
5. Fine-tune with **MoRA** — high-rank updating via a shared rotation matrix  
6. Evaluate both with ROUGE-1/2/L  
7. Save the best checkpoint as `.pth` and the merged weights for inference

---
### Architecture note
`roberta2roberta` is an **encoder-decoder** (seq2seq) model built by tying two
RoBERTa-Large (24-layer) checkpoints together via cross-attention.  
LoRA/MoRA target the `query` and `value` projection matrices inside each
attention block — both in the encoder self-attention and decoder
self/cross-attention layers.

```
Input tokens
     │
  [Encoder: 24 × RoBERTa layers]  ← LoRA/MoRA adapters injected here
     │  context vectors
  [Decoder: 24 × RoBERTa layers]  ← LoRA/MoRA adapters injected here
     │         ↑ cross-attention
  Output token logits
```


In [ ]:
# ── Installations (run once) ──────────────────────────────────────────────
# Uncomment if running in Colab or fresh environment
# !pip install transformers datasets peft rouge-score sentencepiece torch accelerate -q

In [ ]:
import os, math, warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    EncoderDecoderModel,
    RobertaTokenizerFast,
    get_linear_schedule_with_warmup,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel,
)
from rouge_score import rouge_scorer

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR       = "../data"
CHECKPOINT_DIR = "../checkpoints"
MODEL_DIR      = "../models"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Dataset — `torch.utils.data.Dataset`

We subclass `Dataset` and return tokenised tensors.  
The tokeniser converts text → token IDs + attention masks.  
Labels use `-100` for padding positions so `CrossEntropyLoss` ignores them.

In [ ]:
class CNNDailyMailDataset(Dataset):
    """
    Wraps a CSV with 'article' and 'highlights' columns.
    Returns encoder (article) and decoder (highlights) tokenised tensors.
    """

    def __init__(self, csv_path: str, tokenizer, max_src: int = 512, max_tgt: int = 128):
        self.df        = pd.read_csv(csv_path).dropna().reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_src   = max_src
        self.max_tgt   = max_tgt

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        article   = str(self.df.loc[idx, "article"])
        highlight = str(self.df.loc[idx, "highlights"])

        # ── Encode source (article) ──────────────────────────────────────
        src = self.tokenizer(
            article,
            max_length=self.max_src,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # ── Encode target (highlights) ───────────────────────────────────
        # We add BOS at start; the model learns to predict EOS at the end.
        with self.tokenizer.as_target_tokenizer():
            tgt = self.tokenizer(
                highlight,
                max_length=self.max_tgt,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )

        # Labels: replace padding token id with -100 so loss ignores it
        labels = tgt["input_ids"].squeeze().clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":             src["input_ids"].squeeze(),
            "attention_mask":        src["attention_mask"].squeeze(),
            "decoder_input_ids":     tgt["input_ids"].squeeze(),
            "decoder_attention_mask": tgt["attention_mask"].squeeze(),
            "labels":                labels,
        }

In [ ]:
# ── Load tokeniser ────────────────────────────────────────────────────────
MODEL_NAME = "google/roberta2roberta_L-24_cnn_daily_mail"
print("Loading tokeniser…")
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
# roberta2roberta uses the same tokeniser for both sides

# ── Build datasets ────────────────────────────────────────────────────────
train_ds = CNNDailyMailDataset(f"{DATA_DIR}/cnn_dailymail_train.csv",      tokenizer)
val_ds   = CNNDailyMailDataset(f"{DATA_DIR}/cnn_dailymail_validation.csv", tokenizer)

print(f"Train samples : {len(train_ds):,}")
print(f"Val   samples : {len(val_ds):,}")

# Inspect a sample
sample = train_ds[0]
print("\nSample keys:", list(sample.keys()))
print("input_ids shape:", sample["input_ids"].shape)

In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────
# DataLoader handles:
#   - Batching (collate_fn stacks tensors into (B, seq_len))
#   - Shuffling
#   - Parallel loading (num_workers)

BATCH_SIZE  = 4    # keep small for CPU/MPS; bump to 16+ on A100
NUM_WORKERS = 0    # 0 = main process (safe on macOS)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"Train batches: {len(train_loader):,}")
print(f"Val   batches: {len(val_loader):,}")

## 2. Model wrapper — `nn.Module`

We wrap the HuggingFace `EncoderDecoderModel` inside an `nn.Module` so we
have full control over the forward pass — useful when adding custom heads,
auxiliary losses, or swapping in adapters later.

```
SummarisationModel (nn.Module)
└── backbone  (EncoderDecoderModel)
    ├── encoder  (RobertaModel × 24 layers)
    └── decoder  (RobertaForCausalLM × 24 layers)
```

In [ ]:
class SummarisationModel(nn.Module):
    """
    Thin nn.Module wrapper around EncoderDecoderModel.
    Keeping the HF model as `self.backbone` lets us easily:
      - inject PEFT adapters (LoRA / MoRA)
      - freeze/unfreeze layers
      - export weights
    """

    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = EncoderDecoderModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32,   # use float16 on GPU
        )
        # Set special token IDs (required for generation)
        self.backbone.config.decoder_start_token_id = tokenizer.bos_token_id
        self.backbone.config.eos_token_id           = tokenizer.eos_token_id
        self.backbone.config.pad_token_id           = tokenizer.pad_token_id

    def forward(
        self,
        input_ids,
        attention_mask,
        decoder_input_ids,
        decoder_attention_mask,
        labels=None,
    ):
        """
        Returns HF Seq2SeqLMOutput (contains .loss and .logits).
        When labels are provided the backbone computes cross-entropy loss
        internally (teacher-forcing: decoder sees ground-truth tokens shifted right).
        """
        return self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            labels=labels,
        )

    @torch.no_grad()
    def generate_summary(self, input_ids, attention_mask, max_new_tokens=128):
        """Greedy / beam-search decoding for inference."""
        return self.backbone.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

## 3. LoRA — Low-Rank Adaptation

**Core idea**: Instead of updating the full weight matrix W ∈ ℝ^(d×k),
we learn two small matrices A ∈ ℝ^(d×r) and B ∈ ℝ^(r×k) where r ≪ min(d,k).

```
   W′ = W₀ + ΔW  where  ΔW = B·A   (rank-r decomposition)
          ↑ frozen         ↑ learned (only ~0.1% of params)
```

**Why it works**: The weight changes during fine-tuning live on a low-rank
manifold — LoRA exploits this by parameterising only that subspace.

Key hyperparams:
- `r` — rank (4–64 typical). Higher r = more capacity but more params.
- `lora_alpha` — scaling factor: ΔW is scaled by α/r  
- `lora_dropout` — regularisation
- `target_modules` — which weight matrices to adapt (q_proj, v_proj most common)

In [ ]:
def build_lora_model(base_model_name: str, r: int = 16) -> SummarisationModel:
    """
    Load base model and inject LoRA adapters.
    Only ~1% of parameters become trainable.
    """
    model = SummarisationModel(base_model_name)

    # For EncoderDecoderModel with RoBERTa we target attention projections.
    # roberta attention uses: query, value (and key, dense) projections.
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=r,                             # rank
        lora_alpha=r * 2,               # scaling = alpha/r = 2 (standard)
        target_modules=["query", "value"],  # adapt Q and V in every attn layer
        lora_dropout=0.05,
        bias="none",                     # don't adapt biases
        inference_mode=False,
    )

    # get_peft_model wraps backbone and freezes all non-LoRA params
    model.backbone = get_peft_model(model.backbone, lora_cfg)
    model.backbone.print_trainable_parameters()
    return model

print("Building LoRA model…")
lora_model = build_lora_model(MODEL_NAME, r=16)
lora_model = lora_model.to(DEVICE)

## 4. MoRA — High-Rank Updating via Shared Rotation

**Paper**: *MoRA: High-Rank Updating for Parameter-Efficient Fine-Tuning* (2024)

**Key difference from LoRA**:  
LoRA learns ΔW = BA (rank-r). With the same parameter budget MoRA instead
learns a **square** matrix M ∈ ℝ^(r×r) and uses fixed (non-learned) compression
and decompression functions to map it back to the full dimension:

```
   ΔW = decompress( M ) · compress( · )
         ↑ fixed rotation   ↑ fixed projection
         M is square → can represent full-rank updates within budget
```

**Why better?** Tasks that require memorisation (e.g., instruction following,
domain-specific facts) benefit from high-rank updates. MoRA achieves this
without increasing parameter count vs LoRA at the same r.

Below we implement MoRA as a custom `nn.Module` adapter.

In [ ]:
class MoRALinear(nn.Module):
    """
    MoRA adapter wrapping a frozen Linear layer.

    For a weight W ∈ ℝ^(out_features × in_features):
      - We allocate a square matrix M ∈ ℝ^(r × r)  (same param budget as LoRA)
      - compress:   x ∈ ℝ^(in)  → ℝ^r  via fixed random projection A
      - core:       z ∈ ℝ^r     → ℝ^r  via learned M
      - decompress: z ∈ ℝ^r     → ℝ^(out) via fixed random projection B
      - The original linear path is kept frozen; we ADD the adapter output.
    """

    def __init__(self, base_layer: nn.Linear, r: int = 16, alpha: float = 32.0):
        super().__init__()
        self.base   = base_layer          # frozen original weights
        self.r      = r
        self.scale  = alpha / r

        in_f  = base_layer.in_features
        out_f = base_layer.out_features

        # Fixed (non-learned) compression / decompression matrices
        # We use the same random seed so they're deterministic.
        g = torch.Generator().manual_seed(42)
        A = torch.randn(r, in_f,  generator=g) / math.sqrt(in_f)
        B = torch.randn(out_f, r, generator=g) / math.sqrt(r)
        self.register_buffer("A", A)   # not a parameter, not updated by optimizer
        self.register_buffer("B", B)

        # Learned square core matrix M ∈ ℝ^(r × r)  ← the key MoRA insight
        self.M = nn.Parameter(torch.zeros(r, r))

        # Freeze base layer
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        # Original frozen path
        base_out = self.base(x)                   # (B, seq, out)

        # MoRA adapter path: compress → core → decompress
        z   = x @ self.A.T                         # (B, seq, r)   compress
        z   = z @ self.M.T                         # (B, seq, r)   learned core
        out = z @ self.B.T                         # (B, seq, out) decompress

        return base_out + self.scale * out


def inject_mora(model: nn.Module, r: int = 16, alpha: float = 32.0,
                target_names=("query", "value")) -> nn.Module:
    """
    Walk the model and replace any Linear layer whose name contains
    one of `target_names` with a MoRALinear wrapper.
    """
    replaced = 0
    for name, module in list(model.named_modules()):
        for target in target_names:
            if target in name and isinstance(module, nn.Linear):
                # Navigate to parent and replace attribute
                *parent_parts, child_name = name.split(".")
                parent = model
                for part in parent_parts:
                    parent = getattr(parent, part)
                setattr(parent, child_name, MoRALinear(module, r=r, alpha=alpha))
                replaced += 1
                break

    total  = sum(p.numel() for p in model.parameters())
    train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"MoRA: replaced {replaced} layers | trainable params: {train:,} / {total:,} "
          f"({100*train/total:.2f}%)")
    return model


print("Building MoRA model…")
mora_model_base = SummarisationModel(MODEL_NAME)
mora_model      = inject_mora(mora_model_base, r=16, alpha=32.0)
mora_model      = mora_model.to(DEVICE)

## 5. Training Loop

Standard PyTorch training loop:
```
for epoch in range(N):
    for batch in train_loader:
        optimizer.zero_grad()
        loss = model(**batch).loss
        loss.backward()               # compute gradients
        optimizer.step()              # update adapter weights only
        scheduler.step()              # warm-up + linear decay
```

We use a shared `train_one_model()` function for both LoRA and MoRA.

In [ ]:
def train_one_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 3,
    lr: float = 3e-4,
    tag: str = "model",
    max_steps: int = 200,   # cap steps for quick demo; set None for full training
) -> dict:
    """
    Generic training loop. Returns dict with loss history.
    Saves best checkpoint to CHECKPOINT_DIR/{tag}_best.pth
    """
    # Only optimise trainable (adapter) parameters
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.01)

    total_steps   = min(len(train_loader) * epochs, max_steps or 10**9)
    warmup_steps  = total_steps // 10
    scheduler     = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history       = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    global_step   = 0

    for epoch in range(1, epochs + 1):
        # ── Train ─────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        for step, batch in enumerate(train_loader):
            if max_steps and global_step >= max_steps:
                break

            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            optimizer.zero_grad()
            out  = model(**batch)
            loss = out.loss
            loss.backward()

            # Gradient clipping prevents exploding gradients
            nn.utils.clip_grad_norm_(trainable, max_norm=1.0)

            optimizer.step()
            scheduler.step()

            epoch_loss  += loss.item()
            global_step += 1

            if step % 20 == 0:
                lr_now = scheduler.get_last_lr()[0]
                print(f"  [{tag}] epoch={epoch} step={step}/{len(train_loader)} "
                      f"loss={loss.item():.4f} lr={lr_now:.2e}")

        avg_train = epoch_loss / max(step + 1, 1)
        history["train_loss"].append(avg_train)

        # ── Validate ──────────────────────────────────────────────────────
        model.eval()
        val_loss  = 0.0
        val_steps = 0
        with torch.no_grad():
            for batch in val_loader:
                if val_steps >= 50:  # partial validation for speed
                    break
                batch    = {k: v.to(DEVICE) for k, v in batch.items()}
                out      = model(**batch)
                val_loss += out.loss.item()
                val_steps += 1

        avg_val = val_loss / max(val_steps, 1)
        history["val_loss"].append(avg_val)
        print(f"  [{tag}] epoch={epoch} train_loss={avg_train:.4f} val_loss={avg_val:.4f}")

        # ── Checkpoint ────────────────────────────────────────────────────
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            ckpt_path     = f"{CHECKPOINT_DIR}/{tag}_best.pth"
            # Save adapter weights + optimizer state
            torch.save({
                "epoch":      epoch,
                "model_state": model.state_dict(),
                "optim_state": optimizer.state_dict(),
                "val_loss":   best_val_loss,
                "tag":        tag,
            }, ckpt_path)
            print(f"  ✓ Saved best checkpoint → {ckpt_path}")

    return history

In [ ]:
# ── Train LoRA ────────────────────────────────────────────────────────────
print("=" * 60)
print("Training LoRA…")
print("=" * 60)
lora_history = train_one_model(
    lora_model, train_loader, val_loader,
    epochs=3, lr=3e-4, tag="lora", max_steps=200
)

In [ ]:
# ── Train MoRA ────────────────────────────────────────────────────────────
print("=" * 60)
print("Training MoRA…")
print("=" * 60)
mora_history = train_one_model(
    mora_model, train_loader, val_loader,
    epochs=3, lr=3e-4, tag="mora", max_steps=200
)

## 6. Evaluation — ROUGE Metrics

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) measures overlap
between generated and reference summaries.

| Metric  | What it measures                          |
|---------|-------------------------------------------|
| ROUGE-1 | Unigram overlap (word-level recall)       |
| ROUGE-2 | Bigram overlap (captures local fluency)   |
| ROUGE-L | Longest Common Subsequence (global order) |

Higher = better. CNN/DailyMail SOTA ~ROUGE-2 = 21.

In [ ]:
def evaluate_rouge(model: nn.Module, loader: DataLoader, n_batches: int = 20) -> dict:
    """
    Generate summaries for n_batches of validation data and compute ROUGE.
    """
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    all_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            # Generate predicted summaries
            pred_ids = model.generate_summary(input_ids, attention_mask)
            preds    = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

            # Decode reference summaries
            label_ids = batch["labels"].clone()
            label_ids[label_ids == -100] = tokenizer.pad_token_id
            refs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

            for pred, ref in zip(preds, refs):
                s = scorer.score(ref, pred)
                for k in all_scores:
                    all_scores[k].append(s[k].fmeasure)

    return {k: np.mean(v) for k, v in all_scores.items()}


print("Evaluating LoRA…")
lora_rouge = evaluate_rouge(lora_model, val_loader)
print("Evaluating MoRA…")
mora_rouge = evaluate_rouge(mora_model, val_loader)

print("\n── ROUGE Results ──────────────────────────────")
print(f"{'Metric':<12} {'LoRA':>10} {'MoRA':>10}")
print("-" * 34)
for metric in ["rouge1", "rouge2", "rougeL"]:
    print(f"{metric:<12} {lora_rouge[metric]:>10.4f} {mora_rouge[metric]:>10.4f}")

## 7. Save Models

We save in two formats:
1. **`.pth`** — PyTorch native (adapter weights only — small & portable)
2. **HuggingFace** directory — merged weights for inference / quantisation

For LoRA the `peft` library has `merge_and_unload()` which folds LoRA
matrices back into the base weights so you get a standard HF model.

In [ ]:
# ── Save LoRA adapter weights as .pth ────────────────────────────────────
lora_ckpt = torch.load(f"{CHECKPOINT_DIR}/lora_best.pth", map_location="cpu")
torch.save(lora_ckpt["model_state"], f"{MODEL_DIR}/lora_adapter.pth")
print(f"LoRA adapter saved → {MODEL_DIR}/lora_adapter.pth")

# ── Merge LoRA into base weights & save as HF model ──────────────────────
lora_model.eval()
merged = lora_model.backbone.merge_and_unload()   # returns plain EncoderDecoderModel
merged.save_pretrained(f"{MODEL_DIR}/lora_merged")
tokenizer.save_pretrained(f"{MODEL_DIR}/lora_merged")
print(f"Merged LoRA model saved → {MODEL_DIR}/lora_merged/")

# ── Save MoRA adapter weights as .pth ────────────────────────────────────
mora_ckpt = torch.load(f"{CHECKPOINT_DIR}/mora_best.pth", map_location="cpu")
torch.save(mora_ckpt["model_state"], f"{MODEL_DIR}/mora_adapter.pth")
print(f"MoRA adapter saved → {MODEL_DIR}/mora_adapter.pth")

# MoRA: manually extract adapter M matrices and save
mora_state = {}
for name, module in mora_model.named_modules():
    if isinstance(module, MoRALinear):
        mora_state[name + ".M"] = module.M.data
torch.save(mora_state, f"{MODEL_DIR}/mora_M_matrices.pth")
print(f"MoRA M-matrices saved → {MODEL_DIR}/mora_M_matrices.pth")

In [ ]:
# ── Minimalist Inference ──────────────────────────────────────────────────
# Load merged LoRA model and run inference on a sample article

from transformers import EncoderDecoderModel as EDModel

inf_model  = EDModel.from_pretrained(f"{MODEL_DIR}/lora_merged")
inf_tok    = RobertaTokenizerFast.from_pretrained(f"{MODEL_DIR}/lora_merged")
inf_model.eval()

sample_article = """
NASA's Perseverance rover has discovered signs of ancient river delta on Mars,
suggesting the planet once harboured liquid water on its surface. Scientists
are excited about the implications for potential ancient microbial life.
The rover collected rock samples that will be returned to Earth in the 2030s.
"""

inputs = inf_tok(
    sample_article.strip(),
    return_tensors="pt",
    max_length=512,
    truncation=True,
)

with torch.no_grad():
    out_ids = inf_model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=4,
        early_stopping=True,
    )

summary = inf_tok.decode(out_ids[0], skip_special_tokens=True)
print("Generated summary:", summary)

In [ ]:
# ── Summary of results ───────────────────────────────────────────────────
import json
results = {
    "lora": {"rouge": lora_rouge, "train_loss": lora_history["train_loss"]},
    "mora": {"rouge": mora_rouge, "train_loss": mora_history["train_loss"]},
}
with open(f"{MODEL_DIR}/training_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))